# Dev 2: скоринг 12 городов — предварительный расчёт

Снимок ветки Dev 1: `e364ac7d9ee8307d08bcb2bf64f670ad93658889`.
Режим **exploratory**: четыре критерия. Доля пересадок через Москву отсутствует;
туристический показатель — прокси. Итог нельзя выдавать за полный результат ТЗ.
Методика, контракт данных и ограничения: `src/scoring/README.md`.


In [1]:
from pathlib import Path
import sys
import json
import hashlib
import pandas as pd
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src/scoring/city_scorer.py").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.schemas.contracts import ScoringWeights
from src.scoring.city_scorer import load_features, score_cities, weight_sensitivity

DATA_PATH = ROOT / "src/scoring/reference/dev1_e364ac7.csv"
manifest = json.loads((DATA_PATH.parent / "manifest.json").read_text(encoding="utf-8"))
assert hashlib.sha256(DATA_PATH.read_bytes()).hexdigest() == manifest["csv_sha256"]
features = load_features(DATA_PATH)
print("Источник:", manifest["repository"], manifest["commit"])
print("Число городов:", len(features))
display(features[["iata", "city_name", "population_mln", "airport_pax_mln",
                  "dist_to_mow_km", "competitor_gap_score", "tourism_demand_score"]])

Источник: https://github.com/Prepodavately/RedWings__hackathon e364ac7d9ee8307d08bcb2bf64f670ad93658889
Число городов: 12


,iata,city_name,population_mln,airport_pax_mln,dist_to_mow_km,competitor_gap_score,tourism_demand_score
0,CEK,Челябинск,1.474039,2.167,1497,40.0,34.57
1,GOJ,Нижний Новгород,1.813433,1.400,387,75.0,28.08
2,GSV,Саратов,1.245017,1.064,719,60.0,6.70
3,KUF,Самара,2.096344,3.600,844,78.1,28.74
4,KZN,Казань,1.706065,5.400,730,73.8,63.00
5,MRV,Минеральные Воды,0.648282,4.600,1339,56.8,50.00
6,OMS,Омск,1.201893,1.630,2233,66.7,20.10
7,PEE,Пермь,1.270293,2.005,1141,28.6,39.17
8,SVX,Екатеринбург,1.911571,8.385,1430,73.8,70.63
9,TJM,Тюмень,1.018211,2.760,1699,35.7,65.33


## Формула

Население и `min(расстояние до Москвы, расстояние до SVX)` нормируются min-max в 0–100
по полному лонглисту. При равенстве значений — 50. Два готовых индекса уже находятся
в 0–100 и повторно не нормируются. Веса 30/20/30/20 — экспертные приоритеты контракта.

Для полного режима добавляется доля пересадок в Москве с весом 20%; остальные
четыре веса умножаются на 0,8. Получается 24/16/24/16/20. Требуются источники,
периоды, методы и подтверждение осуществимости работы аэропорта.

Одинаковая формула описывает привлекательность города. Для нового хаба опорный
SVX исключается, для прямой сети он может быть конечной точкой. Это не экономика пары А→Б.


In [2]:
weights = ScoringWeights(0.30, 0.20, 0.30, 0.20)
hub = score_cities(features, weights, mode="exploratory", hypothesis="hub")
p2p = score_cities(features, weights, mode="exploratory", hypothesis="p2p")
print("Эффективные веса:", hub.weights)
for name, result in [("Новый хаб", hub), ("Города прямой сети", p2p)]:
    print(name, "— ПРЕДВАРИТЕЛЬНО")
    display(result.top3[["rank", "iata", "city_name", "total_score", "result_status"]])

Эффективные веса: {'population': 0.3, 'hub_distance': 0.2, 'competitor_gap': 0.3, 'tourism_demand': 0.2}
Новый хаб — ПРЕДВАРИТЕЛЬНО


,rank,iata,city_name,total_score,result_status
0,1,KUF,Самара,70.616349,preliminary
1,2,KZN,Казань,67.462111,preliminary
2,3,GOJ,Нижний Новгород,58.035268,preliminary


Города прямой сети — ПРЕДВАРИТЕЛЬНО


,rank,iata,city_name,total_score,result_status
0,1,KUF,Самара,70.616349,preliminary
1,2,KZN,Казань,67.462111,preliminary
2,3,SVX,Екатеринбург,62.437994,preliminary


## Почему город получил такой балл

Каждый сегмент — вклад критерия, сумма сегментов равна итоговому баллу.
Исходные значения и источники сохранены в таблице. На снимке Dev 1 туризм
выражен индексом зарплаты/авиаподвижности, конкуренция — обратной долей лидера
среди рейсов по заданному набору городов. Эти определения требуют пересмотра
перед окончательной защитой.


In [3]:
import plotly.graph_objects as go

eligible = hub.table.loc[hub.table.eligible].sort_values("total_score")
labels = {"population": "Население", "hub_distance": "Удалённость",
          "competitor_gap": "Ниша по индексу Dev 1", "tourism_demand": "Прокси туризма"}
fig = go.Figure()
for criterion, label in labels.items():
    fig.add_bar(y=eligible.city_name, x=eligible[f"contribution_{criterion}"],
                name=label, orientation="h")
fig.update_layout(barmode="stack", title="Новый хаб: предварительный скоринг четырёх критериев",
                  xaxis_title="Вклад в итог, баллы / 100", yaxis_title="",
                  template="plotly_white", height=520, legend_orientation="h")
fig.show(renderer="plotly_mimetype")

## Устойчивость к весам

2 000 случайных наборов: каждый базовый вес × Uniform(0,8; 1,2), затем нормировка;
seed=42. Данные и границы шкал фиксированы. Частота топ-3 — характеристика этого
эксперимента, не вероятность успеха. Нулевой вес в таком анализе остаётся нулевым.


In [4]:
hub_stability = weight_sensitivity(hub, n_samples=2000, relative_change=0.20, seed=42)
p2p_stability = weight_sensitivity(p2p, n_samples=2000, relative_change=0.20, seed=42)
display(hub_stability)
display(p2p_stability.head(4))
assert hub_stability.top3_frequency.sum() == 3.0
assert p2p_stability.top3_frequency.sum() == 3.0

,iata,city_name,rank,result_status,top3_frequency,best_rank,worst_rank,mean_rank,n_samples
0,KUF,Самара,1,preliminary,1.0,1,1,1.0000,2000
1,KZN,Казань,2,preliminary,1.0,2,2,2.0000,2000
2,GOJ,Нижний Новгород,3,preliminary,1.0,3,3,3.0000,2000
3,OMS,Омск,4,preliminary,0.0,4,5,4.4085,2000
4,MRV,Минеральные Воды,5,preliminary,0.0,4,6,4.6005,2000
5,GSV,Саратов,6,preliminary,0.0,5,7,5.9920,2000
6,UFA,Уфа,7,preliminary,0.0,7,8,7.3150,2000
7,CEK,Челябинск,8,preliminary,0.0,6,9,7.6880,2000
8,TJM,Тюмень,9,preliminary,0.0,8,11,9.0255,2000
9,PEE,Пермь,10,preliminary,0.0,9,11,9.9720,2000


,iata,city_name,rank,result_status,top3_frequency,best_rank,worst_rank,mean_rank,n_samples
0,KUF,Самара,1,preliminary,1.0,1,1,1.0,2000
1,KZN,Казань,2,preliminary,1.0,2,2,2.0,2000
2,SVX,Екатеринбург,3,preliminary,1.0,3,3,3.0,2000
3,GOJ,Нижний Новгород,4,preliminary,0.0,4,4,4.0,2000


## Пробелы в данных

Полный режим не подставляет 0% пересадок или моки. Ошибка ниже ожидаема:
она показывает, почему нельзя объявлять предварительную тройку полным ответом на ТЗ.
В `data_audit.csv` каждой паре город/критерий соответствует строка источника и метода.
На текущем снимке многие поля метаданных не заполнены.


In [5]:
try:
    score_cities(features, weights, mode="case")
except ValueError as error:
    print("Ожидаемый блокер полного режима:", error)
else:
    raise AssertionError("Снимок без данных о пересадках не должен проходить полный режим")

audit = hub.audit
missing = audit.assign(missing_source=audit.src.isna() | audit.src.fillna("").astype(str).str.strip().eq(""))
display(missing.groupby("criterion").agg(rows=("iata", "size"), missing_sources=("missing_source", "sum")))
for limitation in hub.limitations:
    print("•", limitation)

Ожидаемый блокер полного режима: Отсутствует колонка moscow_transfer_share_pct


,rows,missing_sources
criterion,,
airport_feasibility,12,12
airport_pax,12,0
competitor_gap,12,12
hub_distance,12,12
moscow_transfer,12,12
population,12,0
tourism_demand,12,12


• Веса — экспертное предположение; результат условен относительно данных и критериев.
• Рейтинг городов не доказывает спрос, прибыльность маршрутов или победу гипотезы.
• Источники записаны поставщиком данных; наличие ссылки не означает внешнюю проверку.
• ПРЕДВАРИТЕЛЬНО: доля пассажиров с пересадкой в Москве не участвует в формуле.
• Происхождение, сопоставимость периодов и возможности аэропортов требуют проверки.
• tourism_demand_score и competitor_gap_score используются в определении Dev 1; проверьте прокси.


## Что передать команде

Dev 1: добавить O&D-долю московских пересадок, сопоставимый показатель туризма,
методы/периоды, инфраструктуру и точные источники. Поправить смешение 2024/2025 в GOJ.

Dev 4: `score_cities(...)` даёт таблицу, вклады, причины исключений и ограничения;
`get_scored_cities(...)` сохраняет общий DTO. Для показа предварительного расчёта
нужно явно выбрать `mode="exploratory"` и вывести статус рядом с рейтингом.

Капитан: веса являются экспертным допущением. Новый хаб и прямую сеть должен
сравнить финансовый расчёт маршрутов. Отдельно согласовать исходную долю RW:
в материалах около 2,5%, в константах репозитория отношение около 4,10%.

Полные инструкции и журнал использования ИИ — в `src/scoring/README.md` и `AI_USAGE.md`.
